In [ ]:
import pandas as pd

from orion.data import load_signal

# 1. Data

In [ ]:
signal_name = 'S-1'

data = load_signal(signal_name)

data.head()

# 2. Pipeline

In [ ]:
from mlblocks import MLPipeline

pipeline_name = 'matrixprofile'

pipeline = MLPipeline(pipeline_name)

## step by step execution

MLPipelines are compose of a squence of primitives, these primitives apply tranformation and calculation operations to the data and updates the variables within the pipeline. To view the primitives used by the pipeline, we access its `primtivies` attribute. 

The `matrixprofile` contains 7 primitives. we will observe how the `context` (which are the variables held within the pipeline) are updated after the execution of each primitive.

In [ ]:
pipeline.primitives

### time segments aggregate
this primitive creates an equi-spaced time series by aggregating values over fixed specified interval.

* **input**: `X` which is an n-dimensional sequence of values.
* **output**:
    - `X` sequence of aggregated values, one column for each aggregation method.
    - `index` sequence of index values (first index of each aggregated segment).

In [ ]:
context = pipeline.fit(data, output_=0)
context.keys()

In [ ]:
for i, x in list(zip(context['index'], context['X']))[:5]:
    print("entry at {} has value {}".format(i, x))

### SimpleImputer
this primitive is an imputation transformer for filling missing values.
* **input**: `X` which is an n-dimensional sequence of values.
* **output**: `X` which is a transformed version of X.

In [ ]:
step = 1

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

### MinMaxScaler
this primitive transforms features by scaling each feature to a given range.
* **input**: `X` the data used to compute the per-feature minimum and maximum used for later scaling along the features axis.
* **output**: `X` which is a transformed version of X.

In [ ]:
step = 2

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
# after scaling the data between [0, 1]
# in this example, no change is observed
# since the data was pre-handedly scaled

for i, x in list(zip(context['index'], context['X']))[:5]:
    print("entry at {} has value {}".format(i, x))

### reshape

this primitive flattens the array.
* **input**: `X` n-dimensional values.
* **output**: `X` which is a flat version of X.

In [ ]:
step = 3

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
context['X'].shape

### stump

this primitive computes the matrix profile of `X`.
* **input**: `X` n-dimensional values.
* **output**: `y` which is the matrix profile of X.

In [ ]:
step = 4

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
context['y'].shape

### slice array by dim

this primitive extracts the distance to the nearest neighbor from the matrix profile.
* **input**: `y` n-dimensional values.
* **output**: `y` which is the distance array in y.

In [ ]:
step = 5

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
context['y'].shape

### reshape

this primitive flattens the array.
* **input**: `y` n-dimensional values.
* **output**: `errors` which is a flat version of y.

In [ ]:
step = 6

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

### find anomalies

this primitive extracts anomalies from sequences of errors following the approach explained in the [related paper](https://arxiv.org/pdf/1802.04431.pdf).

* **input**: 
    - `errors` array of errors.
    - `index` array of indices of errors.
* **output**: `y` array containing start-index, end-index, score for each anomalous sequence that was found.

In [ ]:
step = 7

context = pipeline.fit(**context, output_=step, start_=step)
context.keys()

In [ ]:
pd.DataFrame(context['y'], columns=['start', 'end', 'severity'])